# Multi-Stock Price Prediction Using LSTM

This notebook extends the single-stock LSTM approach to predict stock prices across
**multiple stocks** (AAPL, GOOGL, MSFT, ORCL, ABNB) using 15-minute intraday data.

Key idea: Per-window z-score normalization makes the model price-agnostic,
so it learns **relative price patterns** that generalize across stocks with different price ranges.

1. What are the price characteristics of each stock?
2. Can a single LSTM model predict prices across multiple stocks?
3. How does per-stock performance compare?
4. What are the common failure cases across stocks?

---
## Getting the Data

We load pre-split multi-stock 15-minute intraday data from Excel files (train / val / test).
Each file contains data for AAPL, GOOGL, MSFT, ORCL, and ABNB.

In [ ]:
import pandas as pd
import numpy as np
import os
import math

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.style.use('fivethirtyeight')
%matplotlib inline

from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ---- PyTorch ----
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ---- Auto-detect environment: Kaggle vs Local ----
if os.path.exists('/kaggle/input'):
    BASE = '/kaggle/input/datasets/jackieveloped/ltsm-stock-dataset'
    print('Environment: Kaggle')
else:
    BASE = '/Users/jackietan/Documents/workshop/academic/csc713m/price_prediction_model/multi-pairs/'
    print('Environment: Local')

# ---- Load pre-split data ----
df_train = pd.read_excel(os.path.join(BASE, 'stock_train.xlsx'))
df_val   = pd.read_excel(os.path.join(BASE, 'stock_val.xlsx'))
df_test  = pd.read_excel(os.path.join(BASE, 'stock_test.xlsx'))

for df in [df_train, df_val, df_test]:
    df['OpenTime'] = pd.to_datetime(df['OpenTime'])

SYMBOLS = sorted(df_train['Symbol'].unique())
print(f'\nStocks: {SYMBOLS}\n')

for name, df in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    print(f'{name:5s} : {len(df):>7,} rows  {df["OpenTime"].min()} → {df["OpenTime"].max()}')
    for sym in SYMBOLS:
        n = (df['Symbol'] == sym).sum()
        print(f'        {sym}: {n:>6,} rows')
    print()

## Descriptive Statistics

In [ ]:
# Price range comparison across stocks
df_all = pd.concat([df_train, df_val, df_test], axis=0).reset_index(drop=True)

summary = df_all.groupby('Symbol')['Close'].agg(['count', 'min', 'max', 'mean', 'std'])
summary.columns = ['Count', 'Min', 'Max', 'Mean', 'Std']
summary['Range'] = summary['Max'] - summary['Min']
print('=== Per-Stock Close Price Summary ===')
print(summary.round(2))
print(f'\nTotal rows: {len(df_all):,}')

## Close Price History

In [ ]:
fig, axes = plt.subplots(len(SYMBOLS), 1, figsize=(16, 4 * len(SYMBOLS)), sharex=True)

val_start  = df_val['OpenTime'].min()
test_start = df_test['OpenTime'].min()

for ax, sym in zip(axes, SYMBOLS):
    sub = df_all[df_all['Symbol'] == sym].set_index('OpenTime')
    ax.plot(sub.index, sub['Close'], linewidth=0.6)
    ax.axvline(val_start,  color='orange', linestyle='--', linewidth=1.2, alpha=0.7)
    ax.axvline(test_start, color='red',    linestyle='--', linewidth=1.2, alpha=0.7)
    ax.set_title(f'{sym} Close Price', fontsize=13)
    ax.set_ylabel('Price ($)')
    ax.grid(True, alpha=0.3)

axes[0].legend(['Close', f'Val start ({val_start.date()})', f'Test start ({test_start.date()})'],
               fontsize=10, loc='upper left')
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

---
# Predicting Log Return Using MsLSTM (Multi-Stock)

Architecture from **"A Multimodal Deep Fusion Method for Stock Movement Prediction"**:
- **Positional Encoding** on input features
- **Dual-branch BiLSTM**: short-term (raw) + long-term (causal avg-pooled)
- **Multi-Head Cross-Attention**: Q=short, K=V=long
- **Fusion**: LayerNorm(Linear(Concat(H_short, H_attended)))
- **Temporal Attention** → fixed-length representation
- **MLP head** → predicted log return (regression)

```
X (B, T, F)
   ├── PosEnc → BiLSTM_short → H_short
   └── PosEnc → CausalAvgPool(k=7) → BiLSTM_long → H_long
                    ↓
   MultiHeadAttn(Q=H_short, K=H_long, V=H_long) → H_attended
                    ↓
   LayerNorm(Linear([H_short ∥ H_attended])) → H_fused
                    ↓
   Temporal Attention → h (B, hidden)
                    ↓
   MLP → ŷ (B,)  predicted log return
```

Key difference from single-stock: **Per-window z-score** normalization makes the model price-agnostic.

In [ ]:
# ---- Feature selection ----
FEATURES = [
    # OHLCV
    'Open', 'High', 'Low', 'Volume',
    # MA distance (only SMA)
    'SMA20Dist', 'SMA50Dist',
    # Returns
    'LogReturn1', 'LogReturn5', 'LogReturn15',
    # Volatility / Momentum
    'Vol10', 'Vol20', 'Vol60',
    'VolumeChange', 'Mom10',
    # Candlestick structure
    'Range',          # log(High/Low)
    'Body', 'UpperWick', 'LowerWick',
    # Volume features
    'LogVolume',      # log(Volume)
    'VolMA20',        # Volume / SMA(Volume,20) - 1
    # Gap
    'Gap',            # log(Open/Close.shift(1))
    # Market context
    'SPY_LogReturn1', 'SPY_LogReturn5',
    'QQQ_LogReturn1', 'QQQ_LogReturn5',
    # 'Close',  # 不放进输入，避免模型学成复制 last price
]
SEQ_LEN = 30
TARGET_COL = 'TargetLogRet1'

# Fill NaN in derived features (first rows per stock)
_fill_cols = [c for c in FEATURES if c not in ('Open','High','Low','Volume')]
for _df in [df_train, df_val, df_test]:
    _df[_fill_cols] = _df[_fill_cols].fillna(0)

print(f'Features: {len(FEATURES)} columns (Close excluded)')
print(f'Sequence length: {SEQ_LEN}')
print(f'Target: {TARGET_COL} = log(C_{{t+1}} / C_t)')
print(f'Normalization: per-window z-score (X only)')
print(f'\nFeature list:')
for i, f in enumerate(FEATURES):
    print(f'  {i:2d}. {f}')

In [ ]:
# ---- Build target: next-step log return ----
df_all_seq = df_all.sort_values(['Symbol', 'OpenTime']).copy()
_fill_cols = [c for c in FEATURES if c not in ('Open','High','Low','Volume')]
df_all_seq[_fill_cols] = df_all_seq[_fill_cols].fillna(0)

for sym in SYMBOLS:
    mask = df_all_seq['Symbol'] == sym
    close = df_all_seq.loc[mask, 'Close']
    df_all_seq.loc[mask, TARGET_COL] = np.log(close.shift(-1) / close)

# ---- Per-window z-score (X only) ----
def per_window_zscore(X, eps=1e-8):
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True)
    return (X - mu) / (sd + eps)

# ---- Build sequences by symbol ----
def build_sequences_by_symbol(df, features, seq_len, target_col, price_col='Close'):
    """
    Returns:
      X:          (N, seq_len, F)  z-scored input features
      y:          (N,)             next-step log return
      prev_close: (N,)             close at time t (for price reconstruction)
      sym_arr:    (N,)             symbol labels
      t_end:      (N,)             OpenTime at time t
    """
    X_list, y_list, pc_list, sym_list, t_list = [], [], [], [], []

    for sym, g in df.groupby('Symbol', sort=False):
        g = g.sort_values('OpenTime').reset_index(drop=True)
        g = g.dropna(subset=features + [target_col, price_col])
        if len(g) <= seq_len:
            continue

        vals       = g[features].to_numpy(dtype=np.float32)
        yvals      = g[target_col].to_numpy(dtype=np.float32)
        close_vals = g[price_col].to_numpy(dtype=np.float32)
        times      = g['OpenTime'].to_numpy()

        for i in range(seq_len - 1, len(g) - 1):
            start = i - (seq_len - 1)
            Xw = per_window_zscore(vals[start:i + 1])

            X_list.append(Xw)
            y_list.append(yvals[i])
            pc_list.append(close_vals[i])
            sym_list.append(sym)
            t_list.append(times[i])

    return (np.stack(X_list).astype(np.float32),
            np.asarray(y_list, dtype=np.float32),
            np.asarray(pc_list, dtype=np.float32),
            np.asarray(sym_list),
            np.asarray(t_list))

print('Building sequences per stock...')
X, y, prev_close, sym_arr, t_end = build_sequences_by_symbol(
    df_all_seq, FEATURES, SEQ_LEN, TARGET_COL
)
print(f'Total sequences: {len(y):,}  X shape: {X.shape}')

# ---- Split by original train/val/test date boundaries ----
val_start  = df_val['OpenTime'].min()
test_start = df_test['OpenTime'].min()
t_dt = pd.to_datetime(t_end)

splits_mask = {
    'train': t_dt < val_start,
    'val':   (t_dt >= val_start) & (t_dt < test_start),
    'test':  t_dt >= test_start,
}

np.random.seed(42)
data = {}
for split, mask in splits_mask.items():
    data[split] = {
        'X':          X[mask],
        'y':          y[mask],
        'prev_close': prev_close[mask],
        'symbols':    sym_arr[mask],
        'dates':      t_end[mask],
        'stats':      [(float(pc), 1.0) for pc in prev_close[mask]],
    }

# Shuffle training set
idx = np.random.permutation(len(data['train']['y']))
for key in ['X', 'y', 'prev_close', 'symbols', 'dates']:
    data['train'][key] = data['train'][key][idx]
data['train']['stats'] = [data['train']['stats'][i] for i in idx]

print(f'\n=== Combined Dataset ===')
for split in ['train', 'val', 'test']:
    print(f'{split:5s}: X={data[split]["X"].shape}  y={data[split]["y"].shape}')
    for sym in SYMBOLS:
        n = (data[split]['symbols'] == sym).sum()
        print(f'        {sym}: {n:>6,}')

print(f'\ny_train range: [{data["train"]["y"].min():.4f}, {data["train"]["y"].max():.4f}]')
print(f'y_train mean:  {data["train"]["y"].mean():.4f}')
print(f'\nTraining set shuffled: Yes')

# ---- PyTorch DataLoaders ----
BATCH_SIZE = 32

class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(StockDataset(data['train']['X'], data['train']['y']),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(StockDataset(data['val']['X'],   data['val']['y']),
                          batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(StockDataset(data['test']['X'],  data['test']['y']),
                          batch_size=BATCH_SIZE, shuffle=False)

print(f'\nDataLoaders: batch_size={BATCH_SIZE}')
print(f'  Train: {len(train_loader)} batches')
print(f'  Val:   {len(val_loader)} batches')
print(f'  Test:  {len(test_loader)} batches')

## Build MsLSTM Model

**MsLSTM** (Multi-Scale LSTM) adapted for regression:
- Dual BiLSTM branches (short-term + long-term via causal avg-pool)
- Multi-Head Cross-Attention (Q=short, K=V=long)
- Temporal Attention → MLP → predicted log return
- Loss: Huber (robust to outliers)

In [ ]:
# ---- MsLSTM Components ----

class CausalAvgPool(nn.Module):
    """Causal average pooling: at time t only uses [t-k+1, t]"""
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size

    def forward(self, x):
        # x: (B, T, F)
        x = x.transpose(1, 2)                      # (B, F, T)
        x = F.pad(x, (self.kernel_size - 1, 0))    # left padding
        x = F.avg_pool1d(x, self.kernel_size, stride=1)
        return x.transpose(1, 2)                    # (B, T, F)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:pe[:, 1::2].shape[1]])
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


# ---- MsLSTM Regressor ----

class MsLSTM_Regressor(nn.Module):
    """
    Multi-Scale LSTM adapted for regression (log return prediction).
    Architecture from "A Multimodal Deep Fusion Method for Stock Movement Prediction".
    """
    def __init__(self, input_dim, hidden_dim=128,
                 num_layers=2, kernel_size=7,
                 n_heads=8, mlp_dim=256, dropout=0.31):
        super().__init__()

        # Positional Encoding
        self.pos_enc = PositionalEncoding(input_dim, dropout=dropout)

        # Short-term branch: BiLSTM on raw input
        self.lstm_short = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim // 2,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        # Causal average pooling for long-term smoothing
        self.causal_pool = CausalAvgPool(kernel_size)

        # Long-term branch: BiLSTM on smoothed input
        self.lstm_long = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim // 2,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        # Multi-scale attention: Q=short, K=V=long
        self.ms_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )

        # Fusion: Concat(H_short, H_attended) → Linear → LayerNorm
        self.fusion = nn.Linear(hidden_dim * 2, hidden_dim)
        self.norm   = nn.LayerNorm(hidden_dim)

        # Temporal Attention
        self.attn_W = nn.Linear(hidden_dim, hidden_dim // 2)
        self.attn_v = nn.Linear(hidden_dim // 2, 1, bias=False)

        # MLP regression head (no sigmoid — outputs raw log return)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, mlp_dim),
            nn.BatchNorm1d(mlp_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, mlp_dim // 2),
            nn.BatchNorm1d(mlp_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim // 2, 1)
        )

    def forward(self, x):
        # x: (B, T, input_dim)
        x = self.pos_enc(x)

        # Short-term branch
        H_short, _ = self.lstm_short(x)

        # Long-term branch
        x_smooth   = self.causal_pool(x)
        H_long, _  = self.lstm_long(x_smooth)

        # Cross attention: short queries long
        H_attended, _ = self.ms_attn(
            query=H_short, key=H_long, value=H_long
        )

        # Fusion
        H_fused = self.norm(
            self.fusion(torch.cat([H_short, H_attended], dim=-1))
        )

        # Temporal Attention
        score = self.attn_v(torch.tanh(self.attn_W(H_fused)))
        alpha = torch.softmax(score, dim=1)
        H_final = (alpha * H_fused).sum(dim=1)

        # MLP → predicted log return
        pred = self.mlp(H_final).squeeze(-1)
        return pred, alpha


# ---- Instantiate model ----
INPUT_DIM = len(FEATURES)

model = MsLSTM_Regressor(
    input_dim=INPUT_DIM,
    hidden_dim=128,
    num_layers=2,
    kernel_size=7,
    n_heads=8,
    mlp_dim=256,
    dropout=0.31
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Input dim: {INPUT_DIM}  Seq len: {SEQ_LEN}')
print(f'Model parameters: {total_params:,}')
print(model)

In [ ]:
# ---- Training loop ----

def run_epoch(model, loader, optimizer, criterion, training):
    model.train() if training else model.eval()
    total_loss, n_batches = 0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred, _ = model(X_batch)
            loss = criterion(pred, y_batch)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            n_batches += 1

    return total_loss / n_batches


def train_model(model, train_loader, val_loader,
                lr=0.002545, max_epochs=200, patience=15):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.HuberLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=5, factor=0.5
    )

    best_val_loss, best_state, counter = float('inf'), None, 0
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(1, max_epochs + 1):
        tr_loss = run_epoch(model, train_loader, optimizer, criterion, True)
        va_loss = run_epoch(model, val_loader,   optimizer, criterion, False)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)

        scheduler.step(va_loss)

        if va_loss < best_val_loss:
            best_val_loss = va_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            counter = 0
        else:
            counter += 1

        if epoch % 5 == 0 or counter == 0:
            lr_now = optimizer.param_groups[0]['lr']
            print(f'Epoch {epoch:3d} | Train={tr_loss:.6f}  Val={va_loss:.6f}  '
                  f'LR={lr_now:.6f}  {"*best*" if counter == 0 else ""}')

        if counter >= patience:
            print(f'\nEarly stopping at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    print(f'\nBest Val Loss: {best_val_loss:.6f}')
    return model, history


model, history = train_model(model, train_loader, val_loader)

In [ ]:
# Plot training curve
plt.figure(figsize=(12, 5))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'],   label='Val Loss')
plt.title('Multi-Stock MsLSTM Training Curve (Huber Loss)', fontsize=16)
plt.xlabel('Epoch')
plt.ylabel('Loss (Huber)')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Evaluate on All Splits

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ---- PyTorch batch inference helper ----
def predict_numpy(model, X_np, batch_size=256):
    """Run inference on numpy array, return predictions as numpy."""
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_np), batch_size):
            X_batch = torch.tensor(X_np[i:i+batch_size], dtype=torch.float32).to(device)
            pred, _ = model(X_batch)
            preds.append(pred.cpu().numpy())
    return np.concatenate(preds)

# Inverse transform for log return target
def inverse_logret(pred_logret, y_logret, stats):
    last_close = np.array([base for (base, _std) in stats], dtype=float)
    pred_price   = last_close * np.exp(np.array(pred_logret, dtype=float))
    actual_price = last_close * np.exp(np.array(y_logret, dtype=float))
    return pred_price, actual_price

def calc_metrics(actual, pred, name):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae  = mean_absolute_error(actual, pred)
    r2   = r2_score(actual, pred)
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    print(f'{name:6s}  RMSE={rmse:>10.4f}  MAE={mae:>10.4f}  R²={r2:>8.4f}  MAPE={mape:>6.2f}%')
    return rmse, mae, r2, mape

# Predict + inverse transform for each split
predictions = {}
for split in ['train', 'val', 'test']:
    pred_logret = predict_numpy(model, data[split]['X'])
    pred, actual = inverse_logret(pred_logret, data[split]['y'], data[split]['stats'])
    predictions[split] = {
        'pred': pred, 'actual': actual,
        'pred_logret': pred_logret,
        'actual_logret': data[split]['y'].copy(),
    }

# Overall metrics
print(f'{"":6s}  {"RMSE":>10s}  {"MAE":>10s}  {"R²":>8s}  {"MAPE":>6s}')
print('-' * 60)
for split in ['train', 'val', 'test']:
    calc_metrics(predictions[split]['actual'], predictions[split]['pred'], split.capitalize())

In [ ]:
# ---- Per-Stock Metrics (Test Set) ----
print(f'{"Stock":6s}  {"RMSE":>10s}  {"MAE":>10s}  {"R²":>8s}  {"MAPE":>6s}')
print('-' * 60)

test_syms   = data['test']['symbols']
test_actual = predictions['test']['actual']
test_pred   = predictions['test']['pred']

per_stock_metrics = {}
for sym in SYMBOLS:
    mask = test_syms == sym
    metrics = calc_metrics(test_actual[mask], test_pred[mask], sym)
    per_stock_metrics[sym] = metrics

In [ ]:
# ---- Actual vs Predicted per stock (hidden — set SHOW_PLOTS=True to enable) ----
SHOW_PLOTS = true

if SHOW_PLOTS:
    splits_config = [
        ('train', 'Train', 'steelblue', 'orange'),
        ('val',   'Val',   'blue',      'orange'),
        ('test',  'Test',  'blue',      'red'),
    ]

    for split, split_name, c_actual, c_pred in splits_config:
        fig, axes = plt.subplots(len(SYMBOLS), 1, figsize=(16, 3.5 * len(SYMBOLS)))
        fig.suptitle(f'{split_name} Set — Actual vs Predicted', fontsize=16, y=1.01)

        for ax, sym in zip(axes, SYMBOLS):
            mask = data[split]['symbols'] == sym
            dates  = data[split]['dates'][mask]
            actual = predictions[split]['actual'][mask]
            pred   = predictions[split]['pred'][mask]

            order = np.argsort(dates)
            dates, actual, pred = dates[order], actual[order], pred[order]

            ax.plot(dates, actual, color=c_actual, linewidth=0.8, label='Actual')
            ax.plot(dates, pred,   color=c_pred,   linewidth=0.8, linestyle='--', label='Predicted')
            r2 = r2_score(actual, pred)
            ax.set_title(f'{sym} — {split_name} (R²={r2:.4f})', fontsize=13)
            ax.set_ylabel('Close Price ($)')
            ax.legend(fontsize=10)
            ax.grid(True, alpha=0.3)

        axes[-1].set_xlabel('Date')
        plt.tight_layout()
        plt.show()
else:
    print('Per-stock plots hidden (set SHOW_PLOTS=True to enable)')

In [ ]:
# ---- Scatter Plot per stock (hidden — set SHOW_PLOTS=True to enable) ----
if SHOW_PLOTS:
    fig, axes = plt.subplots(1, len(SYMBOLS), figsize=(4 * len(SYMBOLS), 4))

    for ax, sym in zip(axes, SYMBOLS):
        mask = data['test']['symbols'] == sym
        actual = predictions['test']['actual'][mask]
        pred   = predictions['test']['pred'][mask]

        ax.scatter(actual, pred, alpha=0.3, s=5)
        mn, mx = min(actual.min(), pred.min()), max(actual.max(), pred.max())
        ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5)
        ax.set_title(f'{sym}', fontsize=13)
        ax.set_xlabel('Actual ($)')
        ax.set_ylabel('Predicted ($)')
        r2 = r2_score(actual, pred)
        ax.legend([f'R²={r2:.4f}'], fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.suptitle('Test Set — Actual vs Predicted per Stock', fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('Scatter plots hidden (set SHOW_PLOTS=True to enable)')

## Direction Accuracy

In [ ]:
# ---- Overall Direction Accuracy on Test Set ----
pred_logret_test_all = predict_numpy(model, data['test']['X'])
actual_logret_test_all = data['test']['y']

# Raw direction accuracy (no threshold)
actual_dir_all = np.sign(actual_logret_test_all)
pred_dir_all   = np.sign(pred_logret_test_all)
dir_acc_raw_all = np.mean(actual_dir_all == pred_dir_all)

# Filtered direction accuracy (exclude dead-zone)
RET_TH = 0.001
sig_mask_all = np.abs(actual_logret_test_all) >= RET_TH
dir_acc_filt_all = np.mean(actual_dir_all[sig_mask_all] == pred_dir_all[sig_mask_all])
coverage_all = np.mean(sig_mask_all)

# Up / Down breakdown
up_mask_all   = actual_dir_all > 0
down_mask_all = actual_dir_all < 0
up_recall_all   = np.mean(pred_dir_all[up_mask_all] > 0)
down_recall_all = np.mean(pred_dir_all[down_mask_all] < 0)

print('=' * 60)
print(' Overall Direction Accuracy — TEST Set')
print('=' * 60)
print(f'  Samples:            {len(actual_logret_test_all):,}')
print(f'  DirAcc (raw):       {dir_acc_raw_all:.2%}')
print(f'  DirAcc (filtered):  {dir_acc_filt_all:.2%}  (|log ret| >= {RET_TH}, coverage={coverage_all:.1%})')
print(f'  Up Recall:          {up_recall_all:.2%}  (N={int(up_mask_all.sum()):,})')
print(f'  Down Recall:        {down_recall_all:.2%}  (N={int(down_mask_all.sum()):,})')
print('=' * 60)

## High-Error Analysis

Identify predictions with large deviation across all stocks and splits.

In [ ]:
# Combine all splits for error analysis
all_actual  = np.concatenate([predictions[s]['actual'] for s in ['train', 'val', 'test']])
all_pred    = np.concatenate([predictions[s]['pred']   for s in ['train', 'val', 'test']])
all_dates   = np.concatenate([data[s]['dates']   for s in ['train', 'val', 'test']])
all_symbols = np.concatenate([data[s]['symbols'] for s in ['train', 'val', 'test']])
all_split   = np.concatenate([np.full(len(data[s]['y']), s.capitalize()) for s in ['train', 'val', 'test']])
all_stats   = data['train']['stats'] + data['val']['stats'] + data['test']['stats']

all_error = np.abs(all_actual - all_pred)
prev_close = np.array([base for base, std in all_stats])

# Direction check
actual_dir = np.sign(all_actual - prev_close)
pred_dir   = np.sign(all_pred - prev_close)
dir_correct = actual_dir == pred_dir

threshold = all_error.mean() + 2 * all_error.std()

outlier_df = pd.DataFrame({
    'Date':       all_dates,
    'Symbol':     all_symbols,
    'Split':      all_split,
    'Prev Close': np.round(prev_close, 4),
    'Close':      np.round(all_actual, 4),
    'Predicted':  np.round(all_pred, 4),
    'Error':      np.round(all_error, 4),
    'Actual Dir': ['Up' if d > 0 else 'Down' if d < 0 else 'Flat' for d in actual_dir],
    'Pred Dir':   ['Up' if d > 0 else 'Down' if d < 0 else 'Flat' for d in pred_dir],
    'Correct':    ['Y' if c else 'N' for c in dir_correct],
})

dt = pd.to_datetime(outlier_df['Date'])
outlier_df['Weekday'] = dt.dt.day_name()
outlier_df['Time']    = dt.dt.strftime('%H:%M')
outlier_df['ET Time'] = (dt - pd.Timedelta(hours=5)).dt.strftime('%H:%M')

outlier_df = outlier_df[outlier_df['Error'] > threshold].sort_values('Error', ascending=False)
outlier_df.set_index('Date', inplace=True)

print(f'Threshold: |error| > {threshold:.4f}  (mean + 2\u03c3)')
print(f'Found {len(outlier_df)} bars with large deviation:\n')

# Direction accuracy among outliers
correct_count = (outlier_df['Correct'] == 'Y').sum()
print(f'=== Direction Accuracy (outliers) ===')
print(f'  Correct: {correct_count}/{len(outlier_df)} = {correct_count/len(outlier_df):.1%}')
print()

# Per-stock distribution
print('=== Stock Distribution ===')
print(outlier_df['Symbol'].value_counts().to_string())
print()

# Split distribution
print('=== Split Distribution ===')
print(outlier_df['Split'].value_counts().to_string())
print()

outlier_df[['Symbol', 'Split', 'Prev Close', 'Close', 'Predicted', 'Error',
            'Actual Dir', 'Pred Dir', 'Correct', 'Weekday', 'Time', 'ET Time']].head(100)

In [ ]:
RET_TH = 0.001  # dir acc dead-zone: 0.1%

# ---- predict return (PyTorch) ----
pred_logret_test = predict_numpy(model, data['test']['X'])

# ---- invert return -> price for visualization/diagnostics ----
prev_close_test = data['test']['prev_close']
y_test = data['test']['y']
sym_test = data['test']['symbols']

pred_price_test = prev_close_test * np.exp(pred_logret_test)
true_price_test = prev_close_test * np.exp(y_test)

# ===== Diagnostics helpers =====
def r2_score_np(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - ss_res / (ss_tot + 1e-12)

def corr_np(a, b):
    a = np.asarray(a).reshape(-1)
    b = np.asarray(b).reshape(-1)
    if len(a) < 2:
        return np.nan
    return np.corrcoef(a, b)[0, 1]

def dir_acc_with_return_threshold_from_price(y_true_price, y_pred_price, ret_th=0.001):
    y_true_price = np.asarray(y_true_price).reshape(-1)
    y_pred_price = np.asarray(y_pred_price).reshape(-1)
    true_ret = np.log((y_true_price[1:] + 1e-12) / (y_true_price[:-1] + 1e-12))
    pred_ret = np.log((y_pred_price[1:] + 1e-12) / (y_true_price[:-1] + 1e-12))

    mask = np.abs(true_ret) >= ret_th
    coverage = np.mean(mask)
    if np.sum(mask) == 0:
        return np.nan, coverage
    return np.mean(np.sign(true_ret[mask]) == np.sign(pred_ret[mask])), coverage

def dir_acc_no_th_from_price(y_true_price, y_pred_price):
    y_true_price = np.asarray(y_true_price).reshape(-1)
    y_pred_price = np.asarray(y_pred_price).reshape(-1)
    true_dir = np.sign(y_true_price[1:] - y_true_price[:-1])
    pred_dir = np.sign(y_pred_price[1:] - y_true_price[:-1])
    return np.mean(true_dir == pred_dir)

def make_results_by_symbol(y_true_price, y_pred_price, sym_arr):
    res = {}
    for s in np.unique(sym_arr):
        m = (sym_arr == s)
        res[s] = {"y_true_price": y_true_price[m], "y_pred_price": y_pred_price[m]}
    return res

def make_diagnostics_table(results_by_symbol, ret_th=0.001):
    rows = []
    for sym, d in results_by_symbol.items():
        y_true = np.asarray(d["y_true_price"]).reshape(-1)
        y_pred = np.asarray(d["y_pred_price"]).reshape(-1)

        r2_model = r2_score_np(y_true, y_pred)

        # persistence baseline: y_hat(t) = y_true(t-1)
        y_persist = np.r_[y_true[0], y_true[:-1]]
        r2_persist = r2_score_np(y_true, y_persist)

        dir_no_th = dir_acc_no_th_from_price(y_true, y_pred)
        dir_ret_th, cov = dir_acc_with_return_threshold_from_price(y_true, y_pred, ret_th=ret_th)

        true_dir = np.sign(y_true[1:] - y_true[:-1])
        prev_dir = np.r_[true_dir[0], true_dir[:-1]]
        dir_prev_baseline = np.mean(true_dir == prev_dir)

        rows.append({
            "symbol": sym,
            "R2_model": r2_model,
            "R2_persist": r2_persist,
            "DirAcc_no_th": dir_no_th,
            "DirAcc_ret_th": dir_ret_th,
            "Coverage_ret_th": cov,
            "DirAcc_prevDir_baseline": dir_prev_baseline,
            "corr(pred,actual)": corr_np(y_pred, y_true),
            "corr(pred,actual_shift1)": corr_np(y_pred[1:], y_true[:-1]),
        })

    return pd.DataFrame(rows).sort_values("symbol").reset_index(drop=True)

# ===== Build results and show table =====
results_test = make_results_by_symbol(true_price_test, pred_price_test, sym_test)

print("=== Diagnostics: return-target MsLSTM (TEST) ===")
display(make_diagnostics_table(results_test, ret_th=RET_TH))

## Per-Stock Feature Importance (Permutation Importance)

For each stock, we shuffle one feature at a time across the sequence dimension and measure the increase in MSE on the test set. A large increase means the feature is important for that stock's prediction.

In [ ]:
# ---- Permutation Importance per Stock ----

def permutation_importance_per_stock(model, data_split, features, symbols_list,
                                     n_repeats=5, batch_size=256, seed=42):
    """
    For each stock and each feature, shuffle that feature column across samples
    and measure the increase in MSE (log-return space).
    Returns: dict[symbol] -> DataFrame with columns [feature, base_mse, perm_mse, importance, rank]
    """
    rng = np.random.RandomState(seed)
    X_all = data_split['X']
    y_all = data_split['y']
    syms  = data_split['symbols']

    results = {}

    for sym in symbols_list:
        mask = syms == sym
        X_sym = X_all[mask].copy()
        y_sym = y_all[mask]

        # Base MSE
        pred_base = predict_numpy(model, X_sym, batch_size=batch_size)
        base_mse = np.mean((y_sym - pred_base) ** 2)

        importances = []
        for feat_idx, feat_name in enumerate(features):
            perm_mses = []
            for _ in range(n_repeats):
                X_perm = X_sym.copy()
                # Shuffle this feature across all samples (keep time structure within each sample)
                perm_idx = rng.permutation(len(X_perm))
                X_perm[:, :, feat_idx] = X_sym[perm_idx, :, feat_idx]

                pred_perm = predict_numpy(model, X_perm, batch_size=batch_size)
                perm_mses.append(np.mean((y_sym - pred_perm) ** 2))

            mean_perm_mse = np.mean(perm_mses)
            importances.append({
                'feature': feat_name,
                'base_mse': base_mse,
                'perm_mse': mean_perm_mse,
                'importance': mean_perm_mse - base_mse,  # increase in MSE
                'importance_pct': (mean_perm_mse - base_mse) / (base_mse + 1e-12) * 100,
            })

        df_imp = pd.DataFrame(importances).sort_values('importance', ascending=False).reset_index(drop=True)
        df_imp['rank'] = range(1, len(df_imp) + 1)
        results[sym] = df_imp

    return results


print('Computing permutation importance per stock (this may take a few minutes)...')
imp_results = permutation_importance_per_stock(
    model, data['test'], FEATURES, SYMBOLS, n_repeats=5
)

# ---- Display ALL features per stock, sorted by importance (descending) ----
pd.set_option('display.max_rows', None)

for sym in SYMBOLS:
    df_imp = imp_results[sym]
    print(f'\n{"="*70}')
    print(f' {sym} — All Features Ranked by Importance (Test Set)')
    print(f' Base MSE: {df_imp["base_mse"].iloc[0]:.8f}')
    print(f'{"="*70}')
    display(df_imp[['rank', 'feature', 'importance', 'importance_pct']])

# ---- Features that are NOT useful (importance <= 0) per stock ----
print(f'\n\n{"="*70}')
print(' Features with NO positive importance (can consider removing)')
print(f'{"="*70}')
for sym in SYMBOLS:
    df_imp = imp_results[sym]
    useless = df_imp[df_imp['importance'] <= 0]['feature'].tolist()
    if useless:
        print(f'  {sym}: {", ".join(useless)}')
    else:
        print(f'  {sym}: (all features contribute positively)')

In [ ]:
# ---- Heatmap: Feature Importance across all Stocks ----

# Build importance matrix (stocks x features)
imp_matrix = pd.DataFrame({
    sym: imp_results[sym].set_index('feature')['importance_pct']
    for sym in SYMBOLS
}).T

# Sort features by average importance
feat_order = imp_matrix.mean(axis=0).sort_values(ascending=False).index
imp_matrix = imp_matrix[feat_order]

fig, ax = plt.subplots(figsize=(18, 6))
sns.heatmap(imp_matrix, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'MSE Increase (%)'})
ax.set_title('Feature Importance by Stock (% MSE Increase when Shuffled)', fontsize=15)
ax.set_ylabel('Stock')
ax.set_xlabel('Feature')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ---- Summary: universally useful vs stock-specific features ----
print('\n=== Feature Usefulness Summary ===\n')
avg_imp = imp_matrix.mean(axis=0).sort_values(ascending=False)
print('Universally Important (avg importance_pct > 1%):')
for feat, val in avg_imp[avg_imp > 1].items():
    print(f'  {feat:20s}  avg +{val:.1f}%')

print('\nUniversally Useless (avg importance_pct <= 0):')
for feat, val in avg_imp[avg_imp <= 0].items():
    print(f'  {feat:20s}  avg {val:.1f}%')

print('\nStock-Specific (useful for some, not others):')
for feat in avg_imp.index:
    vals = imp_matrix[feat]
    if vals.max() > 1 and vals.min() < 0:
        helpful = [s for s in SYMBOLS if vals[s] > 1]
        harmful = [s for s in SYMBOLS if vals[s] < 0]
        print(f'  {feat:20s}  helps: {", ".join(helpful)}  |  hurts: {", ".join(harmful)}')

## Per-Stock Direction Accuracy (All Splits)

In [ ]:
# ---- Per-Stock Direction Accuracy across all splits ----

RET_TH_DIR = 0.001  # dead-zone threshold for filtered dir acc

rows = []
for split in ['train', 'val', 'test']:
    pred_lr = predict_numpy(model, data[split]['X'])
    y_lr    = data[split]['y']
    syms    = data[split]['symbols']
    pc      = data[split]['prev_close']

    for sym in SYMBOLS:
        mask = syms == sym
        pred_logret_sym = pred_lr[mask]
        actual_logret_sym = y_lr[mask]
        prev_close_sym = pc[mask]

        # Direction from log return: positive log return = up
        actual_dir = np.sign(actual_logret_sym)
        pred_dir   = np.sign(pred_logret_sym)

        # 1) Raw direction accuracy (no threshold)
        n_total = len(actual_dir)
        n_correct = np.sum(actual_dir == pred_dir)
        dir_acc_raw = n_correct / n_total

        # 2) Direction accuracy excluding dead-zone (|actual log return| < threshold)
        sig_mask = np.abs(actual_logret_sym) >= RET_TH_DIR
        n_sig = np.sum(sig_mask)
        if n_sig > 0:
            dir_acc_filtered = np.mean(actual_dir[sig_mask] == pred_dir[sig_mask])
            coverage = n_sig / n_total
        else:
            dir_acc_filtered = np.nan
            coverage = 0.0

        # 3) Up/Down breakdown
        up_mask   = actual_dir > 0
        down_mask = actual_dir < 0
        flat_mask = actual_dir == 0

        up_acc   = np.mean(pred_dir[up_mask] > 0) if up_mask.sum() > 0 else np.nan
        down_acc = np.mean(pred_dir[down_mask] < 0) if down_mask.sum() > 0 else np.nan

        rows.append({
            'Split': split.capitalize(),
            'Symbol': sym,
            'N': n_total,
            'DirAcc_raw': dir_acc_raw,
            'DirAcc_filtered': dir_acc_filtered,
            'Coverage': coverage,
            'Up_Recall': up_acc,
            'Down_Recall': down_acc,
            'N_up': int(up_mask.sum()),
            'N_down': int(down_mask.sum()),
            'N_flat': int(flat_mask.sum()),
        })

df_dir = pd.DataFrame(rows)

# ---- Display ALL rows, sorted by DirAcc_filtered descending ----
pd.set_option('display.max_rows', None)

df_dir_display = df_dir.copy()
df_dir_display = df_dir_display.sort_values(['Split', 'DirAcc_filtered'], ascending=[True, False])

for col in ['DirAcc_raw', 'DirAcc_filtered', 'Coverage', 'Up_Recall', 'Down_Recall']:
    df_dir_display[col] = df_dir_display[col].map(lambda x: f'{x:.2%}' if not np.isnan(x) else 'N/A')

for split in ['Train', 'Val', 'Test']:
    sub = df_dir_display[df_dir_display['Split'] == split].copy()
    sub = sub.set_index('Symbol')

    print(f'\n{"="*70}')
    print(f' {split} Set — Direction Accuracy per Stock (sorted by DirAcc_filtered ↓)')
    print(f' (filtered threshold: |log return| >= {RET_TH_DIR})')
    print(f'{"="*70}')

    display(sub[['N', 'DirAcc_raw', 'DirAcc_filtered', 'Coverage',
                 'Up_Recall', 'Down_Recall', 'N_up', 'N_down', 'N_flat']])

    # Overall for this split
    all_pred_lr = predict_numpy(model, data[split.lower()]['X'])
    all_actual_lr = data[split.lower()]['y']
    overall_raw = np.mean(np.sign(all_actual_lr) == np.sign(all_pred_lr))
    sig = np.abs(all_actual_lr) >= RET_TH_DIR
    overall_filt = np.mean(np.sign(all_actual_lr[sig]) == np.sign(all_pred_lr[sig])) if sig.sum() > 0 else np.nan
    print(f'  Overall {split}: DirAcc_raw={overall_raw:.2%}  DirAcc_filtered={overall_filt:.2%}')

# ---- Cross-split comparison: sorted by Test DirAcc_filtered ----
print(f'\n\n{"="*70}')
print(' Cross-Split Comparison (sorted by Test DirAcc_filtered ↓)')
print(f'{"="*70}')

pivot = df_dir.pivot_table(index='Symbol', columns='Split',
                           values='DirAcc_filtered', aggfunc='first')
pivot = pivot[['Train', 'Val', 'Test']].sort_values('Test', ascending=False)
pivot_fmt = pivot.map(lambda x: f'{x:.2%}' if not np.isnan(x) else 'N/A')
display(pivot_fmt)

# ---- Bar chart comparison (Test set, sorted) ----
test_sub = df_dir[df_dir['Split'] == 'Test'].copy()
test_sub = test_sub.sort_values('DirAcc_filtered', ascending=False).set_index('Symbol')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
syms_sorted = test_sub.index.tolist()

# 1) Raw vs Filtered Dir Acc
ax = axes[0]
x = np.arange(len(syms_sorted))
w = 0.35
ax.bar(x - w/2, test_sub['DirAcc_raw'],      w, label='Raw', color='steelblue')
ax.bar(x + w/2, test_sub['DirAcc_filtered'],  w, label=f'Filtered (|ret|≥{RET_TH_DIR})', color='coral')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(syms_sorted)
ax.set_ylabel('Accuracy')
ax.set_title('Direction Accuracy (Test)')
ax.legend(fontsize=9)
ax.set_ylim(0.3, 0.7)
ax.grid(axis='y', alpha=0.3)

# 2) Up vs Down Recall
ax = axes[1]
ax.bar(x - w/2, test_sub['Up_Recall'],   w, label='Up Recall', color='green')
ax.bar(x + w/2, test_sub['Down_Recall'], w, label='Down Recall', color='red')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(syms_sorted)
ax.set_ylabel('Recall')
ax.set_title('Up vs Down Recall (Test)')
ax.legend(fontsize=9)
ax.set_ylim(0.0, 1.0)
ax.grid(axis='y', alpha=0.3)

# 3) Sample distribution
ax = axes[2]
ax.bar(x - w/2, test_sub['N_up'],   w, label='Up', color='green', alpha=0.7)
ax.bar(x + w/2, test_sub['N_down'], w, label='Down', color='red', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(syms_sorted)
ax.set_ylabel('Count')
ax.set_title('Up vs Down Sample Count (Test)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Per-Stock Direction Accuracy — Test Set (sorted by DirAcc_filtered ↓)', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# ---- Filter: any split >= 52% AND worst split >= 40% ----
BEST_THRESHOLD = 0.52
WORST_THRESHOLD = 0.40
TOP_N = 200

pivot['Max'] = pivot[['Train', 'Val', 'Test']].max(axis=1)
pivot['Min'] = pivot[['Train', 'Val', 'Test']].min(axis=1)
pivot['Avg'] = pivot[['Train', 'Val', 'Test']].mean(axis=1)

qualified = pivot[
    (pivot['Max'] >= BEST_THRESHOLD) &
    (pivot['Min'] >= WORST_THRESHOLD)
].sort_values('Avg', ascending=False).head(TOP_N)

qualified_show = qualified[['Train', 'Val', 'Test', 'Max', 'Min', 'Avg']].copy()
qualified_fmt = qualified_show.map(lambda x: f'{x:.2%}' if not np.isnan(x) else 'N/A')

print(f'\n\n{"="*70}')
print(f' Qualified Pairs:')
print(f'   - At least one split DirAcc_filtered >= {BEST_THRESHOLD:.0%}')
print(f'   - Worst split DirAcc_filtered >= {WORST_THRESHOLD:.0%}')
print(f' (Top {TOP_N}, sorted by Avg ↓)')
print(f'{"="*70}')

if len(qualified) == 0:
    print('\n  No pairs meet this criteria.')
else:
    print(f'\n  Found {len(qualified)} pair(s):\n')
    display(qualified_fmt)
    print(f'\n  Symbols: {", ".join(qualified.index.tolist())}')

# cleanup temp columns
pivot.drop(columns=['Max', 'Min', 'Avg'], inplace=True, errors='ignore')